In [2]:
import os
import pandas as pd

# Set your folder path here
folder_path = r"D:\AML Project\real world data"
output_file = "merged_excel.xlsx"

# Create an ExcelWriter object
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # Loop through all CSV files in the folder
    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            file_path = os.path.join(folder_path, file)
            
            # Read CSV into DataFrame
            df = pd.read_csv(file_path)
            
            # Use file name (without extension) as sheet name
            sheet_name = os.path.splitext(file)[0][:31]  # Excel sheet names max 31 chars
            
            # Write to Excel in a new sheet
            df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"All CSV files have been merged into '{output_file}' with each file as a separate sheet.")


All CSV files have been merged into 'merged_excel.xlsx' with each file as a separate sheet.


In [3]:
df=pd.read_excel("merged_excel.xlsx")

In [4]:
df.head()

,sector,trading_date,open,high,low,close,volume
0,00DS30,01-01-15,1803.06,1845.61,1803.06,1843.18,22754090
1,00DS30,05-01-15,1843.18,1850.94,1834.75,1844.42,20442650
2,00DS30,06-01-15,1844.42,1862.28,1844.42,1859.07,31960500
3,00DS30,07-01-15,1859.07,1868.74,1855.35,1857.15,29783670
4,00DS30,08-01-15,1857.15,1860.43,1843.55,1854.14,29429290


In [11]:
df['sector'].value_counts()

sector
00DS30        244
00DSES        244
00DSEX        244
WMSHIPYARD    244
RNSPIN        244
             ... 
SIMTEX         28
UCB            20
REGENTTEX      13
FUWANGCER       2
ULC             1
Name: count, Length: 345, dtype: int64

In [13]:
# === CONFIG ===
input_excel = "merged_excel.xlsx"   # <- change if needed
output_excel = input_excel          # write back to same file
output_sheet = "master"

# --- helpers ---
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    # 1) tidy column names: strip + lowercase
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]

    # 2) trim leading/trailing spaces from ALL string cells
    # (works even if there are non-string columns)
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # 3) normalize/convert trading_date column to datetime
    # try to find a likely trading_date column and standardize its name
    possible_names = {
        "trading_date", "trading date", "trade_date", "trade date",
        "date", "tradedate"
    }
    cols = set(df.columns)
    target_col = None
    for name in possible_names:
        if name in cols:
            target_col = name
            break
    if target_col and target_col != "trading_date":
        df = df.rename(columns={target_col: "trading_date"})
        target_col = "trading_date"

    if target_col == "trading_date":
        # Convert to datetime; coerce errors to NaT
        df["trading_date"] = pd.to_datetime(
            df["trading_date"], errors="coerce", infer_datetime_format=True
        )

    return df

# --- load all sheets, clean, concat ---
all_sheets = pd.read_excel(input_excel, sheet_name=None)  # dict: {sheet_name: DataFrame}
cleaned_frames = []

for sheet_name, df in all_sheets.items():
    if isinstance(df, pd.DataFrame):
        cleaned = clean_df(df)
        # keep origin if useful (optional—comment out if you don't want it)
        cleaned.insert(0, "source_sheet", sheet_name)
        cleaned_frames.append(cleaned)

# Concatenate
if not cleaned_frames:
    raise ValueError("No sheets found to concatenate.")
master = pd.concat(cleaned_frames, ignore_index=True)

# --- write back as a new/updated sheet ---
# requires pandas >= 1.4 for if_sheet_exists='replace'
with pd.ExcelWriter(output_excel, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    master.to_excel(writer, sheet_name=output_sheet, index=False)

print(f"✅ Wrote cleaned & concatenated data to '{output_excel}' → sheet '{output_sheet}'.")


C:\Users\user\AppData\Local\Temp\ipykernel_17820\3427839576.py:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
C:\Users\user\AppData\Local\Temp\ipykernel_17820\3427839576.py:34: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df["trading_date"] = pd.to_datetime(
C:\Users\user\AppData\Local\Temp\ipykernel_17820\3427839576.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["trading_date"] = pd.to_datetime(
C:\Users\user\AppData\Local\Temp\ipykernel_17820\3427839576.py:14: FutureWarning: DataFrame.applymap has been deprecated. U

✅ Wrote cleaned & concatenated data to 'merged_excel.xlsx' → sheet 'master'.


In [15]:
import os

# pick the column that identifies the company
company_col = "sector"   # <-- change this if your column has a different name

# 1. Get top 5 companies by value counts
top5_companies = master[company_col].value_counts().head(5).index.tolist()
print("Top 5 companies:", top5_companies)

# 2. Filter master to only those companies
top5_df = master[master[company_col].isin(top5_companies)]

# 3. Save to CSV in same directory
output_csv = os.path.join(os.getcwd(), "top5_companies.csv")
top5_df.to_csv(output_csv, index=False)

print(f"Saved dataset with top 5 companies to {output_csv}")


Top 5 companies: ['GREENDELMF', '1JANATAMF', 'DBH1STMF', 'IFILISLMF1', 'PHPMF1']
Saved dataset with top 5 companies to d:\AML Project\real world data\top5_companies.csv
